# Лабораторная работа 5: Графы (Вариант 2)

## Слесарев Никита ФИТ-231


In [1]:
from collections import deque
import heapq

## Реализация класса Graph и Vertex из лекции


In [2]:
class Vertex:
    """Класс вершины графа"""
    def __init__(self, key):
        self.id = key
        self.connectedTo = {}  # словарь соседей: {вершина: вес}
    
    def addNeighbor(self, nbr, weight=0):
        """Добавить соседа с весом"""
        self.connectedTo[nbr] = weight
    
    def getConnections(self):
        """Получить все соединения"""
        return self.connectedTo.keys()
    
    def getId(self):
        """Получить id вершины"""
        return self.id
    
    def getWeight(self, nbr):
        """Получить вес ребра до соседа"""
        return self.connectedTo.get(nbr, None)
    
    def __str__(self):
        return str(self.id) + ' connectedTo: ' + str([x.id for x in self.connectedTo])


class Graph:
    """Класс графа на основе списка смежности"""
    def __init__(self):
        self.vertList = {}  # словарь вершин: {ключ: объект Vertex}
        self.numVertices = 0
    
    def addVertex(self, key):
        """Добавить вершину в граф"""
        self.numVertices += 1
        newVertex = Vertex(key)
        self.vertList[key] = newVertex
        return newVertex
    
    def getVertex(self, n):
        """Получить вершину по ключу"""
        if n in self.vertList:
            return self.vertList[n]
        else:
            return None
    
    def __contains__(self, n):
        """Проверка наличия вершины в графе"""
        return n in self.vertList
    
    def addEdge(self, f, t, weight=0):
        """Добавить направленное ребро от f к t с весом"""
        if f not in self.vertList:
            self.addVertex(f)
        if t not in self.vertList:
            self.addVertex(t)
        self.vertList[f].addNeighbor(self.vertList[t], weight)
    
    def getVertices(self):
        """Получить список всех вершин"""
        return self.vertList.keys()
    
    def __iter__(self):
        """Итератор по вершинам"""
        return iter(self.vertList.values())
    
    def __str__(self):
        result = ""
        for v in self:
            result += str(v) + "\n"
        return result


## Задание 1


In [3]:
def person_is_seller(name):
    """Проверка, является ли человек продавцом манго"""
    return name[0] == 'a'

def search_bfs(graph, start_name):
    """
    Поиск в ширину (BFS) для поиска продавца манго
    Использует класс Graph вместо словаря
    """
    search_queue = deque()
    start_vertex = graph.getVertex(start_name)
    if start_vertex is None:
        return False
    
    # Добавляем соседей начальной вершины в очередь
    for neighbor in start_vertex.getConnections():
        search_queue.append(neighbor)
    
    searched = set()
    
    while search_queue:
        person_vertex = search_queue.popleft()
        person_name = person_vertex.getId()
        
        if person_name not in searched:
            if person_is_seller(person_name):
                print(person_name + " is a mango seller!")
                return True
            else:
                # Добавляем соседей текущего человека в очередь
                for neighbor in person_vertex.getConnections():
                    search_queue.append(neighbor)
                searched.add(person_name)
    
    return False

# Создание графа с использованием класса Graph
g = Graph()

# Добавление вершин и рёбер
g.addEdge("you", "alice")
g.addEdge("you", "bob")
g.addEdge("you", "claire")
g.addEdge("bob", "anuj")
g.addEdge("bob", "peggy")
g.addEdge("alice", "peggy")
g.addEdge("claire", "thom")
g.addEdge("claire", "jonny")

# Добавляем вершины без исходящих рёбер
g.addVertex("anuj")
g.addVertex("peggy")
g.addVertex("thom")
g.addVertex("jonny")

print("Граф (используя класс Graph):")
for vertex in g:
    neighbors = [v.getId() for v in vertex.getConnections()]
    print(f"{vertex.getId()}: {neighbors}")

print("\nПоиск продавца манго:")
search_bfs(g, "you")


Граф (используя класс Graph):
you: ['alice', 'bob', 'claire']
alice: ['peggy']
bob: ['anuj', 'peggy']
claire: ['thom', 'jonny']
anuj: []
peggy: []
thom: []
jonny: []

Поиск продавца манго:
alice is a mango seller!


True

## Задание 2: Построение обращённого графа

Реализуйте программу, которая по заданному в виде списка смежности графу строит обращённый граф (тоже в виде списка смежности). Обращённый граф получается изменением направления всех рёбер исходного графа.


In [4]:
def reverse_graph(original_graph):
    """
    Построение обращённого графа
    :param original_graph: исходный граф (класс Graph)
    :return: обращённый граф (класс Graph)
    """
    reversed_g = Graph()
    
    # Проходим по всем вершинам исходного графа
    for vertex in original_graph:
        vertex_id = vertex.getId()
        
        # Если вершина не имеет соседей, добавляем её в обращённый граф
        if len(list(vertex.getConnections())) == 0:
            reversed_g.addVertex(vertex_id)
        
        # Для каждого ребра (vertex -> neighbor) создаём обратное (neighbor -> vertex)
        for neighbor in vertex.getConnections():
            neighbor_id = neighbor.getId()
            weight = vertex.getWeight(neighbor)
            # Добавляем обратное ребро
            reversed_g.addEdge(neighbor_id, vertex_id, weight)
    
    return reversed_g

# Пример: создание исходного графа
original = Graph()
original.addEdge("A", "B")
original.addEdge("A", "C")
original.addEdge("B", "D")
original.addEdge("C", "D")
original.addEdge("D", "E")

print("Исходный граф:")
for vertex in original:
    neighbors = [v.getId() for v in vertex.getConnections()]
    if neighbors:
        print(f"{vertex.getId()} -> {neighbors}")

# Построение обращённого графа
reversed_graph = reverse_graph(original)

print("\nОбращённый граф:")
for vertex in reversed_graph:
    neighbors = [v.getId() for v in vertex.getConnections()]
    if neighbors:
        print(f"{vertex.getId()} -> {neighbors}")


Исходный граф:
A -> ['B', 'C']
B -> ['D']
C -> ['D']
D -> ['E']

Обращённый граф:
B -> ['A']
C -> ['A']
D -> ['B', 'C']


## Задание 3: Топологическая сортировка через DFS

Напишите программу, которая при помощи поиска в глубину (DFS) проведет топологическую сортировку графа рецепта блинов и выдаст вариант корректной последовательности шагов.


In [5]:
def dfs_topological_sort(graph):
    """
    Топологическая сортировка через DFS
    :param graph: граф (класс Graph)
    :return: список вершин в порядке топологической сортировки
    """
    visited = set()
    result = []
    
    def dfs_visit(vertex):
        """Вспомогательная функция для DFS"""
        visited.add(vertex.getId())
        # Рекурсивно посещаем всех соседей
        for neighbor in vertex.getConnections():
            if neighbor.getId() not in visited:
                dfs_visit(neighbor)
        # Добавляем вершину в результат после обработки всех её соседей
        result.append(vertex.getId())
    
    # Проходим по всем вершинам графа
    for vertex in graph:
        if vertex.getId() not in visited:
            dfs_visit(vertex)
    
    # Разворачиваем список, так как добавляли в конец
    result.reverse()
    return result

# Создание графа рецепта блинов
pancake_graph = Graph()

# Добавление рёбер согласно графу из задания
# Разогреть сковородку -> Смешать ингредиенты -> Выливать смесь -> Перевернуть -> Полить сиропом
pancake_graph.addEdge("Разогреть сковородку", "Смешать ингредиенты")
pancake_graph.addEdge("Смешать ингредиенты", "Выливать смесь")
pancake_graph.addEdge("Выливать смесь", "Перевернуть")
pancake_graph.addEdge("Перевернуть", "Полить сиропом")

# Ингредиенты должны быть смешаны перед выливанием
pancake_graph.addEdge("Яйцо", "Смешать ингредиенты")
pancake_graph.addEdge("Блинная смесь", "Смешать ингредиенты")
pancake_graph.addEdge("Растительное масло", "Смешать ингредиенты")
pancake_graph.addEdge("Молоко", "Смешать ингредиенты")

print("Граф рецепта блинов:")
for vertex in pancake_graph:
    neighbors = [v.getId() for v in vertex.getConnections()]
    if neighbors:
        print(f"{vertex.getId()} -> {neighbors}")

# Топологическая сортировка
sorted_steps = dfs_topological_sort(pancake_graph)

print("\nТопологическая сортировка (корректная последовательность шагов):")
for i, step in enumerate(sorted_steps, 1):
    print(f"{i}. {step}")


Граф рецепта блинов:
Разогреть сковородку -> ['Смешать ингредиенты']
Смешать ингредиенты -> ['Выливать смесь']
Выливать смесь -> ['Перевернуть']
Перевернуть -> ['Полить сиропом']
Яйцо -> ['Смешать ингредиенты']
Блинная смесь -> ['Смешать ингредиенты']
Растительное масло -> ['Смешать ингредиенты']
Молоко -> ['Смешать ингредиенты']

Топологическая сортировка (корректная последовательность шагов):
1. Молоко
2. Растительное масло
3. Блинная смесь
4. Яйцо
5. Разогреть сковородку
6. Смешать ингредиенты
7. Выливать смесь
8. Перевернуть
9. Полить сиропом


## Задание 4 (Вариант 2): Алгоритм Дейкстры для поиска минимального пути

У вас имеется список городов и набор автобусных маршрутов между ними. Маршруты однонаправленные. Пользователь вводит два города. Программа должна определить минимальный по количеству километров возможный путь от первого до второго города.

**Вариант 2:**
- Города: Белово, Ленинск-Кузнецкий, Киселевск, Гурьевск, Мыски, Кемерово, Новосибирск
- Маршруты указаны в коде


In [6]:
def dijkstra_graph(graph, start_key, end_key=None):
    """
    Алгоритм Дейкстры для поиска кратчайшего пути во взвешенном графе
    Использует класс Graph
    :param graph: граф (класс Graph)
    :param start_key: ключ начальной вершины
    :param end_key: ключ конечной вершины (опционально)
    :return: словари с расстояниями и родителями
    """
    start_vertex = graph.getVertex(start_key)
    if start_vertex is None:
        return {}, {}
    
    # Инициализация
    distances = {}
    parents = {}
    pq = [(0, start_key)]  # приоритетная очередь: (расстояние, ключ вершины)
    visited = set()
    
    # Инициализируем расстояния для всех вершин
    for vertex_key in graph.getVertices():
        distances[vertex_key] = float('inf')
    distances[start_key] = 0
    parents[start_key] = None
    
    while pq:
        current_dist, current_key = heapq.heappop(pq)
        
        if current_key in visited:
            continue
        
        visited.add(current_key)
        
        current_vertex = graph.getVertex(current_key)
        if current_vertex is None:
            continue
        
        # Если нашли конечную вершину, можно остановиться
        if end_key and current_key == end_key:
            break
        
        # Обрабатываем соседей текущей вершины
        for neighbor_vertex in current_vertex.getConnections():
            neighbor_key = neighbor_vertex.getId()
            weight = current_vertex.getWeight(neighbor_vertex)
            
            # Инициализируем расстояние для новой вершины, если нужно
            if neighbor_key not in distances:
                distances[neighbor_key] = float('inf')
            
            new_dist = current_dist + weight
            
            if new_dist < distances[neighbor_key]:
                distances[neighbor_key] = new_dist
                parents[neighbor_key] = current_key
                heapq.heappush(pq, (new_dist, neighbor_key))
    
    return distances, parents

def get_path_from_parents(parents, end_key):
    """Восстановление пути по словарю родителей"""
    path = []
    current = end_key
    
    while current is not None:
        path.append(current)
        current = parents.get(current)
    
    return path[::-1]

# Создание графа маршрутов (Вариант 2)
# Расстояния между городами (в километрах, примерные значения)
routes_graph = Graph()

# Маршруты согласно варианту 2
routes_graph.addEdge("Белово", "Ленинск-Кузнецкий", 50)
routes_graph.addEdge("Белово", "Гурьевск", 30)
routes_graph.addEdge("Ленинск-Кузнецкий", "Гурьевск", 25)
routes_graph.addEdge("Ленинск-Кузнецкий", "Мыски", 80)
routes_graph.addEdge("Ленинск-Кузнецкий", "Кемерово", 60)
routes_graph.addEdge("Ленинск-Кузнецкий", "Новосибирск", 200)
routes_graph.addEdge("Киселевск", "Ленинск-Кузнецкий", 40)
routes_graph.addEdge("Киселевск", "Новосибирск", 250)
routes_graph.addEdge("Гурьевск", "Мыски", 70)
routes_graph.addEdge("Мыски", "Кемерово", 90)
routes_graph.addEdge("Новосибирск", "Кемерово", 180)

# Добавляем вершины без исходящих рёбер
for city in ["Белово", "Ленинск-Кузнецкий", "Киселевск", "Гурьевск", "Мыски", "Кемерово", "Новосибирск"]:
    if city not in routes_graph:
        routes_graph.addVertex(city)

print("Граф маршрутов (Вариант 2):")
for vertex in routes_graph:
    neighbors = []
    for neighbor in vertex.getConnections():
        weight = vertex.getWeight(neighbor)
        neighbors.append(f"{neighbor.getId()}({weight}км)")
    if neighbors:
        print(f"{vertex.getId()} -> {', '.join(neighbors)}")


Граф маршрутов (Вариант 2):
Белово -> Ленинск-Кузнецкий(50км), Гурьевск(30км)
Ленинск-Кузнецкий -> Гурьевск(25км), Мыски(80км), Кемерово(60км), Новосибирск(200км)
Гурьевск -> Мыски(70км)
Мыски -> Кемерово(90км)
Новосибирск -> Кемерово(180км)
Киселевск -> Ленинск-Кузнецкий(40км), Новосибирск(250км)


In [7]:
# Пример использования: поиск пути от Белово до Новосибирска
start_city = "Белово"
end_city = "Новосибирск"

print(f"\n{'='*60}")
print(f"Поиск минимального пути от '{start_city}' до '{end_city}'")
print(f"{'='*60}")

distances, parents = dijkstra_graph(routes_graph, start_city, end_city)

if end_city in distances and distances[end_city] != float('inf'):
    path = get_path_from_parents(parents, end_city)
    
    print(f"\nМинимальное расстояние: {distances[end_city]} км")
    print(f"Маршрут: {' -> '.join(path)}")
    
    # Детализация маршрута
    print(f"\nДетализация маршрута:")
    total_distance = 0
    for i in range(len(path) - 1):
        from_city = path[i]
        to_city = path[i + 1]
        from_vertex = routes_graph.getVertex(from_city)
        to_vertex = routes_graph.getVertex(to_city)
        if from_vertex and to_vertex in from_vertex.getConnections():
            distance = from_vertex.getWeight(to_vertex)
            total_distance += distance
            print(f"  {from_city} -> {to_city}: {distance} км")
    
    print(f"\nОбщее расстояние: {total_distance} км")
else:
    print(f"\nПуть от '{start_city}' до '{end_city}' не найден!")



Поиск минимального пути от 'Белово' до 'Новосибирск'

Минимальное расстояние: 250 км
Маршрут: Белово -> Ленинск-Кузнецкий -> Новосибирск

Детализация маршрута:
  Белово -> Ленинск-Кузнецкий: 50 км
  Ленинск-Кузнецкий -> Новосибирск: 200 км

Общее расстояние: 250 км


In [8]:
# Интерактивный ввод городов
print(f"\n{'='*60}")
print("Интерактивный поиск пути")
print(f"{'='*60}")

# Можно раскомментировать для интерактивного ввода:
# start_input = input("Введите начальный город: ")
# end_input = input("Введите конечный город: ")

# Для демонстрации используем пример
start_input = "Белово"
end_input = "Кемерово"

print(f"\nНачальный город: {start_input}")
print(f"Конечный город: {end_input}")

distances, parents = dijkstra_graph(routes_graph, start_input, end_input)

if end_input in distances and distances[end_input] != float('inf'):
    path = get_path_from_parents(parents, end_input)
    
    print(f"\nМинимальное расстояние: {distances[end_input]} км")
    print(f"Маршрут: {' -> '.join(path)}")
    
    # Детализация
    print(f"\nДетализация маршрута:")
    for i in range(len(path) - 1):
        from_city = path[i]
        to_city = path[i + 1]
        from_vertex = routes_graph.getVertex(from_city)
        to_vertex = routes_graph.getVertex(to_city)
        if from_vertex and to_vertex in from_vertex.getConnections():
            distance = from_vertex.getWeight(to_vertex)
            print(f"  {from_city} -> {to_city}: {distance} км")
else:
    print(f"\nПуть от '{start_input}' до '{end_input}' не найден!")



Интерактивный поиск пути

Начальный город: Белово
Конечный город: Кемерово

Минимальное расстояние: 110 км
Маршрут: Белово -> Ленинск-Кузнецкий -> Кемерово

Детализация маршрута:
  Белово -> Ленинск-Кузнецкий: 50 км
  Ленинск-Кузнецкий -> Кемерово: 60 км


In [9]:
# Поиск всех кратчайших путей от начального города
print(f"\n{'='*60}")
print(f"Все кратчайшие пути от '{start_city}' (алгоритм Дейкстры):")
print(f"{'='*60}")

# Запускаем Дейкстру без указания конечной вершины
distances_all, parents_all = dijkstra_graph(routes_graph, start_city)

for city in sorted(distances_all.keys()):
    if city != start_city:
        distance = distances_all[city]
        if distance != float('inf'):
            path_to_city = get_path_from_parents(parents_all, city)
            print(f"\n{start_city} -> {city}:")
            print(f"  Расстояние: {distance} км")
            print(f"  Маршрут: {' -> '.join(path_to_city)}")
        else:
            print(f"\n{start_city} -> {city}: недостижимо")



Все кратчайшие пути от 'Белово' (алгоритм Дейкстры):

Белово -> Гурьевск:
  Расстояние: 30 км
  Маршрут: Белово -> Гурьевск

Белово -> Кемерово:
  Расстояние: 110 км
  Маршрут: Белово -> Ленинск-Кузнецкий -> Кемерово

Белово -> Киселевск: недостижимо

Белово -> Ленинск-Кузнецкий:
  Расстояние: 50 км
  Маршрут: Белово -> Ленинск-Кузнецкий

Белово -> Мыски:
  Расстояние: 100 км
  Маршрут: Белово -> Гурьевск -> Мыски

Белово -> Новосибирск:
  Расстояние: 250 км
  Маршрут: Белово -> Ленинск-Кузнецкий -> Новосибирск
